In [ ]:
!pip install -q torch numpy tiktoken datasets tqdm matplotlib


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [72]:
import torch
from torch import nn
import torch.nn.functional as F

In [73]:
if torch.cuda.is_available():
    device = "cuda"      # NVIDIA GPU (your pod)
elif torch.backends.mps.is_available():
    device = "mps"       # Apple GPU (your Mac)
else:
    device = "cpu"
print(device)

mps


In [74]:
text = open("./input.txt", encoding="utf-8").read()
vocab = sorted(set(text))
id_to_char = {}
char_to_id = {}

for idx, char in enumerate(vocab):
    id_to_char[idx]=char
    char_to_id[char]=idx

def encode(str):
    ret =[]
    for s in str:
        ret.append(char_to_id[s])
    return ret

def decode(list):
    s = ""
    for l in list:
        s+=id_to_char[l]
    return s

def get_batch(split):
    if split == "train": 
        data = train_data
    else: 
        data = test_data
    idx = torch.randint(len(data)-block_size-1, (batch_size,))

    l_x = []
    l_y = []
    for i in idx: 
        l_x.append(data[i: i+block_size])
        l_y.append(data[i+1:i+block_size+1])
        
    x = torch.stack(l_x)
    y = torch.stack(l_y)
    
    x= x.to(device)
    y = y.to(device)
    return x, y

In [75]:
batch_size = 32
block_size = 128
n_embd = 192
n_head = 6
n_layer = 4
dropout = 0.2
lr = 3e-4
steps = 5000

In [76]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [77]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x): 
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1)
        wei = (C ** -0.5 ) * wei
        tril = self.tril[:T, :T]
        wei = wei.masked_fill(tril == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out
        

In [78]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        outs = [h(x) for h in self.heads] 
        outs = torch.cat(outs, dim=-1)
        outs = self.proj(outs)
        outs = self.dropout(outs)
        return outs


In [79]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.seq= nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = self.seq(x)
        x = self.dropout(x)
        return x

In [80]:
class Block(nn.Module):
    def __init__(self, n_head):
        super().__init__()
        self.att = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        out = x + self.att(self.ln1(x))
        out = x + self.ffwd(self.ln2(out))
        return out 

In [81]:
class BigramModel(nn.Module):
        def __init__(self, vocab_size = 65):
                super().__init__()
                self.table = nn.Embedding(65, n_embd)
                self.lm_head = nn.Linear(n_embd, vocab_size)
                self.pos_table = nn.Embedding(block_size, n_embd)
                self.blocks = nn.Sequential(*[Block(n_head) for _ in range(n_layer)])
                self.ln = nn.LayerNorm(n_embd)
                self.dropout = nn.Dropout(dropout)

        def forward(self, idx, targets=None):
                B, T = idx.shape
                tok = self.table(idx)
                pos = self.pos_table(torch.arange(T, device=idx.device))
                x = pos + tok
                x = self.blocks(x)
                x = self.ln(x)
                logits = self.lm_head(x)
                if targets == None: 
                        return logits, None
                else:   
                        _, _, C = logits.shape
                        loss =  F.cross_entropy(logits.view(-1, C), targets.view(-1)) 
                        return logits, loss
        def generate(self, idx, max_new_tokens, block_size = 8):
                for _ in range(max_new_tokens): 
                        logits, _ = self.forward(idx[:, -block_size:])
                        logits = logits[:, -1, :]
                        p = F.softmax(logits, dim=-1)
                        pred = torch.multinomial(p, num_samples=1)
                        idx = torch.cat((idx, pred), dim=1)
                return idx

In [82]:
model = BigramModel().to(device)
x, y = get_batch("train")
logits, loss = model(x, y)

In [83]:
optimizer= torch.optim.AdamW(model.parameters(), lr)
for i in range(steps):
    x, y = get_batch("train")
    optimizer.zero_grad(set_to_none=True)
    logits, loss = model(x, y)
    loss.backward()
    optimizer.step()

    if i%500 == 0:
        print(loss.item())



4.343000411987305
2.4204273223876953
2.1418094635009766
1.9456225633621216
1.7756294012069702
1.7451646327972412
1.6861265897750854
1.7073898315429688
1.5806537866592407
1.537083625793457


In [88]:
@torch.no_grad()
def estimate_loss(): 
    train_loss = 0
    val_loss = 0
    model.eval()

    for _ in range(200):
        x, y = get_batch("train")
        x_v, y_v = get_batch("val")
        _, train_loss_t = model(x, y)
        _, val_loss_t = model(x_v, y_v)
        train_loss+=train_loss_t.item()
        val_loss+=val_loss_t.item()
    model.train()
    return train_loss/200, val_loss/200


In [89]:
estimate_loss()

(1.451628224849701, 1.6467659425735475)

In [94]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)
out = model.generate(start, max_new_tokens=300)
print(decode(out[0].tolist()))


KING LEWIS OF YORK:
The great are to go
As noso prry shall has indeed hath is, I have it of lond,
All we were will de grace.
at what murder'd there no purpilant
That we, bear here? sperpuing kepior his firest, I would how this faver the ablon these of of the sorring seezet brief.

LADY AUR:
Thou ull


In [87]:
sum(p.numel() for p in model.parameters())

1827137

In [92]:
torch.save(model.state_dict(), "shakespeare_medium.pt")

In [93]:
model.load_state_dict(torch.load("shakespeare_medium.pt", map_location=device))

<All keys matched successfully>